## <b><font color='darkblue'>Preface</font></b>
([source](https://developers.googleblog.com/developers-guide-to-multi-agent-patterns-in-adk/)) <font size='3ptx'><b>The world of software development has already learned this lesson: monolithic applications don't scale.</b> Whether you're building a massive e-commerce platform or a complex AI application, relying on a single, all-in-one entity creates bottlenecks, increases debugging costs, and limits specialized performance.</font>

<b>The same principle applies to an AI agent. A single agent tasked with too many responsibilities becomes a "Jack of all trades, master of none."</b> As the complexity of instructions increases, adherence to specific rules degrades, and error rates compound, leading to more and more "hallucinations." If your agent fails, you shouldn't have to tear down the entire prompt to find the bug.

<b>Reliability comes from decentralization and specialization.</b> <b><font color='darkblue'>Multi-Agent Systems</font> (MAS) allow you to build the AI equivalent of a microservices architecture.</b> By assigning specific roles (a Parser, a Critic, a Dispatcher) to individual agents, you build systems that are inherently more modular, testable, and reliable.

<b>In this guide we’ll be using the Google Agent Development Kit (ADK) to illustrate 8 essential design patterns</b>, from the Sequential Pipeline to the Human-in-the-loop design pattern, providing you with the concrete patterns and pseudocode you need to build production-grade agent teams.

### <b><font color='darkgreen'>Agenda</font></b>
- [<b><font size='3ptx'>Sequential Pipeline Pattern (aka the assembly line)</font></b>](#Sequential-Pipeline-Pattern-(aka-the-assembly-line))
- [<b><font size='3ptx'>Coordinator/Dispatcher Pattern (aka the concierge)</font></b>](#Coordinator/Dispatcher-Pattern-(aka-the-concierge))
- [<b><font size='3ptx'>Parallel Fan-Out/Gather Pattern (aka the octopus)</font></b>](#Parallel-Fan-Out/Gather-Pattern-(aka-the-octopus))
- [<b><font size='3ptx'>Hierarchical Decomposition Pattern (aka the russian doll)</font></b>](#Hierarchical-Decomposition-Pattern-(aka-the-russian-doll))
- [<b><font size='3ptx'>Generator and Critic Pattern (aka the editor's desk)</font></b>](#Generator-and-Critic-Pattern-(aka-the-editor's-desk))
- [<b><font size='3ptx'>Iterative Refinement Pattern (aka the sculptor)</font></b>](#Iterative-Refinement-Pattern-(aka-the-sculptor))
- [<b><font size='3ptx'>Human-in-the-Loop Pattern (the human safety net)</font></b>](#Human-in-the-Loop-Pattern-(the-human-safety-net))
- [<b><font size='3ptx'>Composite Patterns (the mix-and-match)</font></b>](#Composite-Patterns-(the-mix-and-match))
- [<b><font size='3ptx'>Best Practices & Summary</font></b>](#Best-Practices-&-Summary)

### <b><font color='darkgreen'>Import Packages</font></b>
Let's import some helpers to facilate demonstrating the agent development here.

In [1]:
import os
import agent_tools

# Suppress duplicate API key warning from google-genai
if "GOOGLE_API_KEY" in os.environ and "GEMINI_API_KEY" in os.environ:
  del os.environ["GEMINI_API_KEY"]

AgentRunner = agent_tools.AgentRunner

## <b><font color='darkblue'>Sequential Pipeline Pattern </font>(aka the assembly line)</b>
<font size='3ptx'><b>Let’s start with the bread and butter of agent workflows.</b> Think of this pattern as a classic assembly line where Agent A finishes a task and hands the baton directly to Agent B. It is linear, deterministic, and refreshingly easy to debug because you always know exactly where the data came from.</font>

<img src="images/seqential_pattern.original.png" width="70%"/>

This is your go-to architecture for data processing pipelines. In the example below, we see <b>a flow for processing raw documents</b>: a <b>Parser Agent</b> turns a raw PDF into text, an <b>Extractor Agent</b> pulls out structured data, and a <b>Summarizer Agent</b> generates the final synopsis.

<b><font size='3ptx'>In ADK, the [SequentialAgent](https://adk.dev/api-reference/python/google-adk.html#google.adk.agents.SequentialAgent) primitive handles the orchestration for you</font></b>. The secret sauce here is state management: simply use the output_key to write to the shared session.state so the next agent in the chain knows exactly where to pick up the work.

### <b><font color='darkgreen'>Thinking, Querying and Summary</font></b>
This module implements a sequential multi-agent assembly line:
1. **`thinking_agent`**: Formulates an effective Google search query from the user's question.
2. **`query_agent`**: Executes the Google search query and retrieves the top 3 results in text.
3. **`summary_agent`**: Synthesizes the search results into a concise, accurate summary.

```python
# Step 1: Thinking Agent (Formulates Google Search query from the user question)
thinking_agent = LlmAgent(
    name="ThinkingAgent",
    model=MODEL_NAME,
    instruction=(
        "You are an expert search query formulation assistant. "
        "Analyze the user's question and compose an effective, concise Google search query. "
        "Output ONLY the search query string without any introductory or explanatory text."
    ),
    description="Composes optimized Google search queries based on user questions.",
    output_key="search_query",
)

# Step 2: Query Agent (Executes Google Search and retrieves top 3 results in text)
query_agent = LlmAgent(
    name="QueryAgent",
    model=MODEL_NAME,
    instruction=(
        "You are an information retrieval agent. Using the search query: {search_query}, "
        "conduct a Google search using the Google Search tool and retrieve the top 3 most "
        "relevant search results in text, including key details and snippets. "
        "Output ONLY the retrieved search results."
    ),
    description="Conducts Google search and retrieves top 3 text search results.",
    tools=[google_search],
    output_key="search_results",
)

# Step 3: Summary Agent (Summarizes the search results to answer the question)
summary_agent = LlmAgent(
    name="SummaryAgent",
    model=MODEL_NAME,
    instruction=(
        "You are an expert summarization assistant. Based on the retrieved search results:\n"
        "{search_results}\n\n"
        "Provide a clear, accurate, and comprehensive summary that directly answers the user's original question."
    ),
    description="Summarizes retrieved search results into a comprehensive answer.",
)

# Orchestrate the Assembly Line using SequentialAgent
pipeline = SequentialAgent(
    name="SearchAndSummarizePipeline",
    sub_agents=[thinking_agent, query_agent, summary_agent],
    description="A sequential pipeline that formulates search queries, retrieves top Google search results, and synthesizes a final summary.",
)

```

In [2]:
import sequential_pipeline_pattern_ex1

runner = AgentRunner(agent=sequential_pipeline_pattern_ex1.pipeline)

In [3]:
question = "Give me examples of agent design patterns."
response = runner.prompt(question)
print(response)


▶ [Root Agent: SearchAndSummarizePipeline] Input
  • User Prompt: Give me examples of agent design patterns.

------------------------------------------------------------
▶ [Sub-Agent: ThinkingAgent] Input
------------------------------------------------------------
  • User Prompt: Give me examples of agent design patterns.

------------------------------------------------------------
◀ [Sub-Agent: ThinkingAgent] Output
------------------------------------------------------------
  • Generated Response:
"AI agent design patterns" examples
  • State Updates:
    - search_query: "AI agent design patterns" examples

------------------------------------------------------------
▶ [Sub-Agent: QueryAgent] Input
------------------------------------------------------------
  • User Prompt: Give me examples of agent design patterns.
  • Input State:
    - search_query: "AI agent design patterns" examples

------------------------------------------------------------
◀ [Sub-Agent: QueryAgent] Ou

## <b><font color='darkblue'>Coordinator/Dispatcher Pattern</font> (aka the concierge)</b>
<font size='3ptx'><b>Sometimes you don’t need a chain; you need a decision maker.</b> In this pattern, a central intelligent agent acts as a concierge/dispatcher. It analyzes the user's intent and dynamically routes the request to the specialist agent best suited for the job.</font>

This is ideal for customer service systems, multi-domain assistants, or triage desks where requests need domain-specific routing (e.g. <b>Billing Specialist</b> vs <b>Technical Support Specialist</b> vs <b>Sales Specialist</b>).

<img src="images/coordinator_dispatcher_agent_pattern.original.png" width="70%"/>

### <b><font color='darkgreen'>LLM-Driven Delegation with AutoFlow</font></b>
In ADK, you define a parent `LlmAgent` and provide a list of specialist `sub_agents`. The ADK AutoFlow mechanism handles automatic transfer (`transfer_to_agent`) based on the descriptions of the child agents.

```python
# Sub-Agent 1: Billing Specialist
billing_specialist = LlmAgent(
    name="BillingSpecialist",
    model="gemini-3.5-flash",
    instruction="You are a Billing Specialist. Assist with invoices, refunds, and subscriptions.",
    description="Handles billing inquiries, invoices, payments, refunds, and subscriptions.",
)

# Sub-Agent 2: Technical Support Specialist
tech_support_specialist = LlmAgent(
    name="TechSupportSpecialist",
    model="gemini-3.5-flash",
    instruction="You are a Technical Support Specialist. Troubleshoot bugs, API issues, and outages.",
    description="Troubleshoots technical issues, error logs, API bugs, and platform connectivity.",
)

# Root Coordinator / Dispatcher
coordinator = LlmAgent(
    name="CoordinatorAgent",
    model="gemini-3.5-flash",
    instruction="Analyze user intent. Route billing issues to BillingSpecialist and bugs to TechSupportSpecialist.",
    description="Main concierge routing user requests to specialized sub-agents.",
    sub_agents=[billing_specialist, tech_support_specialist],
)
```

In [ ]:
import coordinator_dispatcher_pattern_ex2


runner = AgentRunner(agent=coordinator_dispatcher_pattern_ex2.coordinator_agent)
question = "I was charged twice on my credit card for invoice #INV-2025-884."
print(f"User Prompt: {question}\n")
response = runner.prompt(question)
print("\nAgent Response:\n")
print(response)

## <b><font color='darkblue'>Parallel Fan-Out/Gather Pattern</font> (aka the octopus)</b>
<font size='3ptx'><b>Speed matters. If you have tasks that don’t depend on each other, why run them one by one?</b> In this pattern, multiple agents execute tasks simultaneously to reduce latency or gain diverse perspectives. Their outputs are then aggregated by a final synthesizer agent.</font>

This is ideal for automated code reviews, multi-perspective audits, market analysis, or document fact-checking across multiple independent dimensions.

<img src="images/parallel_fan-out.original.png" width="70%"/>

### <b><font color='darkgreen'>Concurrent Execution with ParallelAgent</font></b>
In ADK, `ParallelAgent` runs sub-agents simultaneously. Each sub-agent writes its output to a unique `output_key` in the shared `session.state`. A subsequent `LlmAgent` gathers these outputs to produce a consolidated summary.

```python
# 1. Parallel Specialist Agents
security_scanner = LlmAgent(
    name="SecurityAuditor",
    instruction="Check for vulnerabilities like injection attacks.",
    output_key="security_report",
)
style_checker = LlmAgent(
    name="StyleEnforcer",
    instruction="Check for PEP8 compliance and formatting issues.",
    output_key="style_report",
)
complexity_analyzer = LlmAgent(
    name="PerformanceAnalyst",
    instruction="Analyze time complexity and resource usage.",
    output_key="performance_report",
)

# 2. Fan-Out (ParallelAgent Swarm)
parallel_reviews = ParallelAgent(
    name="CodeReviewSwarm",
    sub_agents=[security_scanner, style_checker, complexity_analyzer],
)

# 3. Gather / Synthesizer Agent
pr_summarizer = LlmAgent(
    name="PRSummarizer",
    instruction="Create a consolidated Pull Request review using {security_report}, {style_report}, and {performance_report}.",
)

# 4. Wrap into a sequential pipeline
workflow = SequentialAgent(sub_agents=[parallel_reviews, pr_summarizer])
```

In [ ]:
import parallel_fan_out_pattern_ex3


runner = AgentRunner(agent=parallel_fan_out_pattern_ex3.workflow)
code_to_review = """def fetch_user_orders(user_id):
    query = 'SELECT * FROM orders WHERE customer_id = ' + str(user_id)
    results = db.execute(query)
    return results
"""
prompt = (
    "Please review the following code snippet for our pull request:\n\n```python\n"
    f"{code_to_review}```"
)
print(f"User Prompt:\n{prompt}\n")
response = runner.prompt(prompt)
print("\nAgent Response:\n")
print(response)

## <b><font color='darkblue'>Hierarchical Decomposition Pattern</font> (aka the russian doll)</b>
<font size='3ptx'><b>Sometimes a task is too big for a single agent's context window.</b> High-level agents can break down complex goals into sub-tasks and delegate them. Unlike simple routing, the parent agent may delegate just <i>part</i> of a task and wait for the result to continue its own reasoning.</font>

This pattern is widely used in executive report generation, autonomous planning, and enterprise workflows where high-level goals are broken down into sub-agent operations.

<img src="images/hierarchical_task_decomposition_agent.original.png" width="70%"/>

### <b><font color='darkgreen'>Sub-Agents as Callable Tools via AgentTool</font></b>
In ADK, you can treat a complete sub-agent hierarchy as a callable tool by wrapping it in `AgentTool(sub_agent)`. The parent agent calls the sub-agent just like any standard Python function or API tool.

```python
# Level 3: Leaf Tool Agents
knowledge_retriever = LlmAgent(
    name="KnowledgeRetrievalAgent",
    description="Searches internal technical database for domain facts.",
    tools=[query_technical_database],
)
fact_summarizer = LlmAgent(
    name="FactSummarizerAgent",
    description="Condenses raw technical text into structured key points.",
)

# Level 2: Sub-Orchestrator Coordinator
research_assistant = LlmAgent(
    name="ResearchAssistant",
    description="Coordinates technical information retrieval and summarization.",
    sub_agents=[knowledge_retriever, fact_summarizer],
)

# Level 1: Top-Level Executive Agent
report_writer = LlmAgent(
    name="ExecutiveReportWriter",
    instruction="Write a comprehensive executive report. Use the ResearchAssistant to gather verified info.",
    tools=[AgentTool(research_assistant)],  # Sub-agent hierarchy exposed as a tool
)
```

In [ ]:
import hierarchical_decomposition_pattern_ex4


runner = AgentRunner(agent=hierarchical_decomposition_pattern_ex4.report_writer)
question = "Provide an executive briefing on recent advances in quantum computing architectures."
print(f"User Prompt: {question}\n")
response = runner.prompt(question)
print("\nAgent Response:\n")
print(response)

## <b><font color='darkblue'>Generator and Critic Pattern</font> (aka the editor's desk)</b>
<font size='3ptx'><b>Generating high-quality, reliable output often requires a second pair of eyes.</b> In this pattern, you separate the creation of content from the validation of content. One agent acts as the Generator, producing a draft, while a second agent acts as the Critic, reviewing it against specific criteria or logical constraints.</font>

This pattern features conditional looping: if the review passes, the loop terminates. If it fails, specific feedback is routed back to the Generator to produce a compliant draft. It is essential for SQL generation, code compilation checks, and regulatory compliance.

<img src="images/generator_critic_agent_pattern.original.png" width="70%"/>

### <b><font color='darkgreen'>Quality Gates with LoopAgent and exit_loop</font></b>
In ADK, `LoopAgent` encapsulates the Generator and Critic sub-agents. The Critic is equipped with the `exit_loop` tool to break out of the loop as soon as the quality gate conditions are satisfied.

```python
# 1. The Generator Agent
sql_generator = LlmAgent(
    name="SQLGenerator",
    instruction="Generate ANSI SQL. If feedback exists in state ({critic_feedback}), fix the errors.",
    output_key="generated_sql",
)

# 2. The Critic Agent (Equipped with exit_loop)
sql_critic = LlmAgent(
    name="SQLCritic",
    instruction="Validate {generated_sql}. If correct and safe, CALL exit_loop. Otherwise explain errors.",
    tools=[exit_loop],
    output_key="critic_feedback",
)

# 3. The Validation Loop
validation_loop = LoopAgent(
    name="SQLValidationLoop", sub_agents=[sql_generator, sql_critic], max_iterations=3
)
```

In [ ]:
import generator_critic_pattern_ex5


runner = AgentRunner(agent=generator_critic_pattern_ex5.validation_loop)
request = "Find top 5 customers with highest total order amount in 2025 who purchased in 'Electronics'."
print(f"User Request: {request}\n")
response = runner.prompt(request)
print("\nAgent Response:\n")
print(response)

## <b><font color='darkblue'>Iterative Refinement Pattern</font> (aka the sculptor)</b>
<font size='3ptx'><b>Great work rarely happens in a single draft.</b> Just like a human writer revises, polishes, and edits, agents often need multiple iterations to produce publication-grade material. In this pattern, agents enter a cycle of generating, critiquing, and refining until the output meets a high qualitative standard.</font>

Unlike the binary Pass/Fail of the Generator/Critic pattern, **Iterative Refinement focuses on progressive qualitative elevation** (e.g. adding nuance, depth, technical diagrams, and removing ambiguity).

<img src="images/iterative_refinement_agent_pattern_1.original.png" width="70%"/>

### <b><font color='darkgreen'>LoopAgent + SequentialAgent Orchestration</font></b>
A Generator produces the initial rough draft, followed by a `LoopAgent` that pairs a **Critique Agent** (which writes actionable notes and can call `exit_loop` if quality is top-notch) with a **Refinement Agent** (which rewrites and polishes the draft).

```python
# 1. Initial Draft Generator
draft_generator = LlmAgent(
    name="InitialDraftGenerator",
    instruction="Write a concise initial draft on the requested topic.",
    output_key="current_draft",
)

# 2. Critique Agent
critic = LlmAgent(
    name="CritiqueAgent",
    instruction="Review {current_draft}. Provide 2-3 specific qualitative improvements, or CALL exit_loop if top-tier.",
    tools=[exit_loop],
    output_key="critique_notes",
)

# 3. Refinement Agent
refiner = LlmAgent(
    name="RefinementAgent",
    instruction="Rewrite {current_draft} incorporating {critique_notes} to elevate depth and clarity.",
    output_key="current_draft",  # Overwrites draft with polished version
)

# 4. Refinement Loop and Workflow
refinement_loop = LoopAgent(
    name="RefinementLoop", sub_agents=[critic, refiner], max_iterations=2
)
workflow = SequentialAgent(sub_agents=[draft_generator, refinement_loop])
```

In [ ]:
import iterative_refinement_pattern_ex6


runner = AgentRunner(agent=iterative_refinement_pattern_ex6.workflow)
topic = (
    "Explain the CAP theorem for distributed databases, its misconceptions,"
    " and real-world PACELC extensions."
)
print(f"User Topic: {topic}\n")
response = runner.prompt(topic)
print("\nAgent Response:\n")
print(response)

## <b><font color='darkblue'>Human-in-the-Loop Pattern</font> (the human safety net)</b>
<font size='3ptx'><b>AI agents are powerful, but sometimes you need a human in the driver's seat for critical decision-making.</b> In this pattern, agents handle the groundwork, but a human must authorize high-stakes actions—specifically operations that are irreversible or carry significant financial or operational consequences.</font>

Examples include approving wire transfers above financial thresholds, granting security permissions, executing database migrations, or deploying code to production.

<img src="images/human-in-the-loop_pattern.original.png" width="70%"/>

### <b><font color='darkgreen'>Safety Gates and Human Authorization Tools</font></b>
In ADK, human-in-the-loop is implemented via specialized approval tools. When an agent detects a high-risk operation, it invokes the authorization tool, which verifies human sign-off before downstream execution proceeds.

```python
# Custom Approval Tool
def request_human_approval(
    action_description: str, amount_usd: float, recipient: str
) -> dict:
  """Requests human operator authorization for high-stakes actions."""
  # Enforces compliance checks / human sign-off
  return {"status": "APPROVED", "approver": "SecurityOfficer_Admin"}


# 1. Transaction Agent with Approval Tool
transaction_agent = LlmAgent(
    name="TransactionAgent",
    instruction="Process transfers. For amounts > $1,000 USD, YOU MUST call request_human_approval first.",
    tools=[request_human_approval],
    output_key="transaction_result",
)

# 2. Audit and Receipt Agent
audit_agent = LlmAgent(
    name="AuditAndReceiptAgent",
    instruction="Generate formal compliance receipts from {transaction_result}.",
)

# 3. Workflow Pipeline
workflow = SequentialAgent(sub_agents=[transaction_agent, audit_agent])
```

In [ ]:
import human_in_the_loop_pattern_ex7


runner = AgentRunner(agent=human_in_the_loop_pattern_ex7.workflow)
prompt = (
    "Please execute an external wire transfer of $45,000 to vendor 'CloudScale"
    " Solutions' for GPU cluster hosting."
)
print(f"User Prompt: {prompt}\n")
response = runner.prompt(prompt)
print("\nAgent Response:\n")
print(response)

## <b><font color='darkblue'>Composite Patterns</font> (the mix-and-match)</b>
<font size='3ptx'><b>Real-world enterprise applications rarely fit into a single box.</b> In production, you will combine multiple patterns to build resilient, end-to-end multi-agent architectures.</font>

For example, an **Enterprise Customer Support System** might:
1. Use a **Coordinator / Dispatcher** to route requests between Billing and Technical Support.
2. For technical requests, trigger a **Parallel Fan-Out Swarm** to search documentation and retrieve customer infrastructure telemetry simultaneously.
3. Feed findings into a **Generator and Critic Loop** to draft, review, and polish customer-facing responses.
4. Chain the entire technical flow inside a **Sequential Pipeline**.

<img src="images/composite_agent_pattern.original.png" width="70%"/>

### <b><font color='darkgreen'>Architecture Overview</font></b>
```
               ┌───────────────────────────────────────┐
               │    EnterpriseSupportConcierge         │
               │         (CoordinatorAgent)            │
               └──────────────────┬────────────────────┘
                                  │
                 ┌────────────────┴────────────────┐
                 ▼                                 ▼
      ┌───────────────────────┐        ┌─────────────────────────┐
      │   BillingSpecialist   │        │ TechnicalSupportPipeline│
      │      (LlmAgent)       │        │    (SequentialAgent)    │
      └───────────────────────┘        └───────────┬─────────────┘
                                                   │
                            ┌──────────────────────┴──────────────────────┐
                            ▼                                             ▼
                 ┌─────────────────────┐                       ┌─────────────────────┐
                 │  TechResearchSwarm  │                       │ SupportQualityLoop  │
                 │   (ParallelAgent)   │                       │     (LoopAgent)     │
                 └──────────┬──────────┘                       └──────────┬──────────┘
                            │                                             │
                  ┌─────────┴─────────┐                         ┌─────────┴─────────┐
                  ▼                   ▼                         ▼                   ▼
           ┌──────────────┐    ┌──────────────┐          ┌──────────────┐    ┌──────────────┐
           │DocSearchAgent│    │ InfraAgent   │          │TechGenerator │    │ ToneCritic   │
           └──────────────┘    └──────────────┘          └──────────────┘    └──────────────┘
```

In [ ]:
import composite_pattern_ex8


runner = AgentRunner(agent=composite_pattern_ex8.enterprise_concierge)
query = (
    "We are encountering 504 Gateway Timeouts in us-central1 on our backend"
    " API."
)
print(f"Customer Query: {query}\n")
response = runner.prompt(query)
print("\nConcierge Response:\n")
print(response)

## <b><font color='darkblue'>Best Practices & Summary</font></b>

### <b><font color='darkgreen'>Summary of the 8 Multi-Agent Design Patterns</font></b>

| # | Design Pattern | Core ADK Primitive | Primary Use Case |
| :--- | :--- | :--- | :--- |
| **1** | **Sequential Pipeline** | `SequentialAgent` | Deterministic assembly line data transformations & sequential task chains |
| **2** | **Coordinator / Dispatcher** | `LlmAgent` (`sub_agents`) | Intent classification, request triage, and dynamic AutoFlow routing |
| **3** | **Parallel Fan-Out / Gather** | `ParallelAgent` + `SequentialAgent` | Concurrent audits, multi-perspective reviews, and low-latency evaluation |
| **4** | **Hierarchical Decomposition** | `AgentTool` | Complex goal breakdown, nested sub-orchestration, and research delegations |
| **5** | **Generator & Critic** | `LoopAgent` + `exit_loop` | Binary quality gates, syntactic/schema compliance, and self-correction loops |
| **6** | **Iterative Refinement** | `SequentialAgent` + `LoopAgent` | Progressive qualitative elevation, technical polishing, and editorial workflows |
| **7** | **Human-in-the-Loop** | Custom Tool / Safety Callback | Authorization for irreversible operations, financial transfers, and production deploys |
| **8** | **Composite Patterns** | Multi-Primitive Composition | Enterprise multi-tier customer support, autonomous engineering, and complex workflows |

---

### <b><font color='darkgreen'>Key Architectural Takeaways</font></b>
1. **State Management is Vital:** In ADK, `session.state` acts as the shared whiteboard across your agent team. Use explicit, descriptive `output_key` names so downstream agents know exactly where to find input variables.
2. **Precise Agent Descriptions:** When using routing and coordinators, the `description` string of each sub-agent functions as its API contract for LLM delegation.
3. **Start Simple and Compose:** Begin with simple sequential or coordinator pipelines before nesting loops and parallel swarms. Modular microservices are easier to test, debug, and monitor.